In [1]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import pandas as pd

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Data and Model Prep

class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        post = self.data.iloc[idx]['post'] + "The reaction of the addressee is " + self.data.iloc[idx]['oReact'] + ". The attribute of the addressor is " + self.data.iloc[idx]['xAttr'] + " and the intent of the addressor is " + self.data.iloc[idx]['xIntent']
        label = int(self.data.iloc[idx]['offensiveYN'] * 2)  # Convert to 0, 1, or 2
        encoding = self.tokenizer(post, truncation=True, max_length=self.max_length, padding='max_length', return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label)
        }

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=3)

model.roberta.encoder.layer[-1].output.dropout.p = 0.1
model.classifier.dropout.p = 0.1

train_data = pd.read_csv("/content/drive/MyDrive/NLP_2024/Project/Data_Atomic/train_dataset_atomic.csv")
test_data = pd.read_csv("/content/drive/MyDrive/NLP_2024/Project/Data_Atomic/test_dataset_atomic.csv")
val_data = pd.read_csv("/content/drive/MyDrive/NLP_2024/Project/Data_Atomic/val_dataset_atomic.csv")


print(train_data.shape)
print(test_data.shape)
print(val_data.shape)

train_dataset = CustomDataset(train_data, tokenizer, max_length=128)
val_dataset = CustomDataset(val_data, tokenizer, max_length=128)
test_dataset = CustomDataset(test_data, tokenizer, max_length=128)

train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print(train_dataset[0])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 5

from tqdm import tqdm
import time

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    start_time = time.time()

    for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)
    print(f'Training Loss: {train_loss}')

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc=f'Validation', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()

    val_loss /= len(val_dataloader)
    print(f'Validation Loss: {val_loss}')

    end_time = time.time()  # End time of epoch
    epoch_time = end_time - start_time  # Time taken for epoch
    print(f'Epoch {epoch + 1} - Time: {epoch_time:.2f} seconds')

import pickle

model_path = "/content/drive/MyDrive/NLP_2024/Project/Classification/Vanilla/Roberta_Finetune_Classifier"

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

import pickle

model_path = "/content/drive/MyDrive/NLP_2024/Project/Classification/Vanilla/Roberta_Finetune_Classifier.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

# test_preds = []
# test_labels = []
# for batch in test_dataloader:
#     input_ids = batch['input_ids'].to(device)
#     attention_mask = batch['attention_mask'].to(device)
#     labels = batch['labels'].to(device)
#     with torch.no_grad():
#         outputs = model(input_ids, attention_mask=attention_mask)
#         logits = outputs.logits
#     test_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
#     test_labels.extend(labels.cpu().numpy())\
# test accuracy
# test_accuracy = accuracy_score(test_labels, test_preds)

# print(f"Test Accuracy: {test_accuracy}")

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model.to(device)

model.eval()
predictions = []
targets = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()  # Convert labels to numpy array for sklearn metrics

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        _, predicted_classes = torch.max(probabilities, dim=1)

        predictions.extend(predicted_classes.cpu().numpy())
        targets.extend(labels)

print('Accuracy:', accuracy_score(targets, predictions))
print('Precision:', precision_score(targets, predictions, average='weighted'))
print('Recall:', recall_score(targets, predictions, average='weighted'))
print('F1 Score:', f1_score(targets, predictions, average='weighted'))

# Print classification report
print('\nClassification Report:')
class_report = classification_report(targets, predictions)
print(class_report)

report_path = "/content/drive/MyDrive/NLP_2024/Project/Classification/Vanilla/Roberta_Finetune_Class_Report"

with open(report_path, 'wb') as f:
    pickle.dump(class_report, f)

report_path = "/content/drive/MyDrive/NLP_2024/Project/Classification/Vanilla/Roberta_Finetune_Class_Report.pkl"

with open(report_path, 'wb') as f:
    pickle.dump(class_report, f)

# model.eval()
# predictionst = []
# targetst = []

# with torch.no_grad():
#     for batch in train_dataloader:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['labels'].cpu().numpy()  # Convert labels to numpy array for sklearn metrics

#         outputs = model(input_ids, attention_mask=attention_mask)
#         logits = outputs.logits
#         probabilities = torch.softmax(logits, dim=1)
#         _, predicted_classes = torch.max(probabilities, dim=1)

#         predictionst.extend(predicted_classes.cpu().numpy())
#         targetst.extend(labels)

# print('Accuracy:', accuracy_score(targetst, predictionst))
# print('Precision:', precision_score(targetst, predictionst, average='weighted'))
# print('Recall:', recall_score(targetst, predictionst, average='weighted'))
# print('F1 Score:', f1_score(targetst, predictionst, average='weighted'))

# # Print classification report
# print('\nClassification Report:')
# print(classification_report(targetst, predictionst))

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


(28863, 25)
(3801, 25)
(3773, 25)
{'input_ids': tensor([    0, 13963,   787,  1215, 43551,   347, 30529,    35,    38,   437,
         7013,    14,   103,     9,  1423,   108,  1250,   828,  5559,   120,
         5283, 34197,   142,    22, 33272,   797,   359,  3914,   131,   563,
          741, 13866,   113,    32,  2375,   359, 10431,  1092,  4531,  3103,
          131,   947, 10431,  1092,  2940,  4419,   131,   947, 10431,  6750,
         3414,   131,   947, 10431, 16316,  3416,   131,   133,  4289,     9,
            5,  1606,  1535,  7048,    16,    45, 10404,     4,    20, 21643,
            9,     5,  1100,   368,    16,    45, 10404,     8,     5,  5927,
            9,     5,  1100,   368,    16,    45, 10404,     2,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,  

Training Loss: 0.6323237536532367


Validation Loss: 0.5030169877720903
Epoch 1 - Time: 898.10 seconds


Training Loss: 0.5217136711376759


...

Validation Loss: 0.520164525309988
Epoch 3 - Time: 897.71 seconds


Training Loss: 0.38220200406743277


Validation Loss: 0.5458910649526919
Epoch 4 - Time: 897.14 seconds


Training Loss: 0.3231089282773142


Validation Loss: 0.6183064323903005
Epoch 5 - Time: 897.06 seconds
Accuracy: 0.8308339910549856
Precision: 0.804921610983935
Recall: 0.8308339910549856
F1 Score: 0.8125177478560608

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.83      0.86      1834
           1       0.16      0.03      0.05       217
           2       0.80      0.93      0.86      1750

    accuracy                           0.83      3801
   macro avg       0.61      0.60      0.59      3801
weighted avg       0.80      0.83      0.81      3801



# Inference

In [2]:
# Data and Model Prep

class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        post = self.data.iloc[idx]['post'] + "The reaction of the addressee is " + self.data.iloc[idx]['oReact'] + ". The attribute of the addressor is " + self.data.iloc[idx]['xAttr'] + " and the intent of the addressor is " + self.data.iloc[idx]['xIntent']
        label = int(self.data.iloc[idx]['offensiveYN'] * 2)  # Convert to 0, 1, or 2
        encoding = self.tokenizer(post, truncation=True, max_length=self.max_length, padding='max_length', return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label)
        }

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=3)

model.roberta.encoder.layer[-1].output.dropout.p = 0.1
model.classifier.dropout.p = 0.1

train_data = pd.read_csv("/content/drive/MyDrive/NLP_2024/Project/Data_Atomic/train_dataset_atomic.csv")
test_data = pd.read_csv("/content/drive/MyDrive/NLP_2024/Project/Data_Atomic/test_dataset_atomic.csv")
val_data = pd.read_csv("/content/drive/MyDrive/NLP_2024/Project/Data_Atomic/val_dataset_atomic.csv")


print(train_data.shape)
print(test_data.shape)
print(val_data.shape)

train_dataset = CustomDataset(train_data, tokenizer, max_length=128)
val_dataset = CustomDataset(val_data, tokenizer, max_length=128)
test_dataset = CustomDataset(test_data, tokenizer, max_length=128)

train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print(train_dataset[0])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(28863, 25)
(3801, 25)
(3773, 25)
{'input_ids': tensor([    0, 13963,   787,  1215, 43551,   347, 30529,    35,    38,   437,
         7013,    14,   103,     9,  1423,   108,  1250,   828,  5559,   120,
         5283, 34197,   142,    22, 33272,   797,   359,  3914,   131,   563,
          741, 13866,   113,    32,  2375,   359, 10431,  1092,  4531,  3103,
          131,   947, 10431,  1092,  2940,  4419,   131,   947, 10431,  6750,
         3414,   131,   947, 10431, 16316,  3416,   131,   133,  4289,     9,
            5,  1606,  1535,  7048,    16,    45, 10404,     4,    20, 21643,
            9,     5,  1100,   368,    16,    45, 10404,     8,     5,  5927,
            9,     5,  1100,   368,    16,    45, 10404,     2,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,  

In [3]:
import pickle

model_path = "/content/drive/MyDrive/NLP_2024/Project/Classification/Vanilla/Roberta_Finetune_Classifier.pkl"

with open(model_path, 'rb') as f:
    modelloaded = pickle.load(f)

In [4]:
# test_preds = []
# test_labels = []
# for batch in test_dataloader:
#     input_ids = batch['input_ids'].to(device)
#     attention_mask = batch['attention_mask'].to(device)
#     labels = batch['labels'].to(device)
#     with torch.no_grad():
#         outputs = model(input_ids, attention_mask=attention_mask)
#         logits = outputs.logits
#     test_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
#     test_labels.extend(labels.cpu().numpy())\
# test accuracy
# test_accuracy = accuracy_score(test_labels, test_preds)

# print(f"Test Accuracy: {test_accuracy}")

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model.to(device)

model.eval()
predictions = []
targets = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()  # Convert labels to numpy array for sklearn metrics

        outputs = modelloaded(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        _, predicted_classes = torch.max(probabilities, dim=1)

        predictions.extend(predicted_classes.cpu().numpy())
        targets.extend(labels)

print('Accuracy:', accuracy_score(targets, predictions))
print('Precision:', precision_score(targets, predictions, average='weighted'))
print('Recall:', recall_score(targets, predictions, average='weighted'))
print('F1 Score:', f1_score(targets, predictions, average='weighted'))

# Print classification report
print('\nClassification Report:')
class_report = classification_report(targets, predictions)
print(class_report)

Accuracy: 0.8308339910549856
Precision: 0.804921610983935
Recall: 0.8308339910549856
F1 Score: 0.8125177478560608

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.83      0.86      1834
           1       0.16      0.03      0.05       217
           2       0.80      0.93      0.86      1750

    accuracy                           0.83      3801
   macro avg       0.61      0.60      0.59      3801
weighted avg       0.80      0.83      0.81      3801

